# HCDE 410 — Stage 4: Provisional analysis

**Research status:** AI-assisted first-pass coding of 200 genuine Steam negative reviews. Not final, human-validated research findings. This public notebook reads de-identified labels only, never review text.

## 1. Load anonymized coding table

Run from the GitHub repository root, or from `analysis/` (both locations supported).

In [ ]:
from pathlib import Path
import csv
from collections import Counter
import matplotlib.pyplot as plt

source = Path("data/ai_first_pass_200_codes.csv")
if not source.exists():
    source = Path("../data/ai_first_pass_200_codes.csv")
with source.open(encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))
assert len(rows) == 200, len(rows)
periods = ["base_2017", "rise_fall_2018", "gathering_2019", "frontier_2020", "recent_2026"]
print("First-pass records:", len(rows), "from", source)

## 2. Feedback specificity levels

Codes: F0 general dissatisfaction; F1 specific problem; F2 explicit change request; F3 concrete remedy. These are provisional model labels.

In [ ]:
levels = ["F0", "F1", "F2", "F3"]
counts = Counter(r["feedback_level_AI_first_pass"] for r in rows)
for level in levels:
    print(level, counts[level], f"({100*counts[level]/len(rows):.1f}%)")
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(levels, [counts[x] for x in levels])
ax.set_xlabel("First-pass feedback level")
ax.set_ylabel("Review count")
ax.set_title("Provisional F0–F3 coding, n=200")
plt.tight_layout()
plt.show()

## 3. Game-system topics

Topic labels overlap across a review. The broad `game_other` category is not a single named game system.

In [ ]:
topics = Counter(t.strip() for r in rows for t in r["issue_topics_semicolon"].split(";"))
game_systems = ["game_ai_opponents", "game_pacing_progression", "game_balance",
                "game_city_management", "game_diplomacy", "game_combat_units",
                "game_map_generation", "game_religion"]
fig, ax = plt.subplots(figsize=(8.8, 4.5))
items = list(reversed(game_systems))
ax.barh([t.replace("game_", "").replace("_", " ") for t in items],
        [topics[t] for t in items])
ax.set_xlabel("Reviews mentioning topic (multiple allowed)")
ax.set_title("AI-assisted first-pass game-system themes")
plt.tight_layout()
plt.show()

## 4. Period-stratified comparisons

40 comments per selected period, not proportional to Steam population. Creation dates are **not** proof of DLC ownership or patch exposure, and reviews can be edited later.

In [ ]:
bottom = [0]*len(periods)
fig, ax = plt.subplots(figsize=(9, 4.8))
for level in levels:
    values = [sum(r["period"] == p and r["feedback_level_AI_first_pass"] == level for r in rows) for p in periods]
    ax.bar(["2017", "2018", "2019", "2020", "2026"], values, bottom=bottom, label=level)
    bottom = [x+y for x,y in zip(bottom, values)]
ax.set_ylabel("Number of reviews (40 per period)")
ax.set_title("Provisional specificity by creation-date window")
ax.legend(title="Code", ncol=4)
plt.tight_layout()
plt.show()

## 5. Before final publication

The 200 labels require human verification, including a 40-item blinded independent reviewer worksheet that is **not committed publicly**. After human review, compare disagreements, adjudicate with original text and regenerate charts using the verified label column. Do not call human–AI agreement an inter-rater human reliability estimate. Original review text and Steam review identifiers remain in private source files.